# Azure vision API experiment

Send one or more local CCTV frames to Azure OpenAI and inspect the response.

Requires `.env` with `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, and `AZURE_OPENAI_MODEL`.

In [1]:
from pathlib import Path

from IPython.display import Image, display

from cctv.analysis.azure_vision import analyze_camera_image, analyze_images, save_experiment
from cctv.utils.azure import load_azure_openai_config
from cctv.utils.paths import place_data_dir
from cctv.utils.prompts import load_place_prompt

azure = load_azure_openai_config()
if not azure.is_configured:
    print("Missing Azure OpenAI configuration in .env")
    print("Required: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_MODEL")
else:
    print("Azure OpenAI configuration loaded")
    print(f"Endpoint: {azure.endpoint}")
    print(f"Model: {azure.model}")
    print(f"Request URL: {azure.api_url}")

Azure OpenAI configuration loaded
Endpoint: https://profinitfoundry.services.ai.azure.com/openai/v1
Model: gpt-5.6-luna
Request URL: https://profinitfoundry.services.ai.azure.com/openai/v1/chat/completions


## Pick images

Point `PLACE_ID` at a folder under `camera_images/<place>/`. Change `IMAGE_NAMES` to a subset, or leave it empty to use all JPEGs in that folder.

In [2]:
PLACE_ID = "hybernska"
IMAGE_NAMES: list[str] = []  # e.g. ["camera_202020_20260907_140653_792.jpg"]

image_dir = place_data_dir(PLACE_ID)
all_images = sorted(image_dir.glob("*.jpg"))

if IMAGE_NAMES:
    image_paths = [image_dir / name for name in IMAGE_NAMES]
else:
    image_paths = all_images

missing = [path for path in image_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing images: {missing}")

print(f"Image dir: {image_dir}")
print(f"Using {len(image_paths)} / {len(all_images)} image(s)")
for path in image_paths:
    print(f"  {path.name}")

Image dir: /home/k8s/Work/Thesis/Project/multimodal-cctv-surveilance/camera_images/hybernska
Using 5 / 5 image(s)
  camera_101048_20260907_143634_894.jpg
  camera_101048_20260907_143637_019.jpg
  camera_101048_20260907_143640_721.jpg
  camera_101048_20260907_143642_802.jpg
  camera_101048_20260907_143644_903.jpg


## Prompt

`load_place_prompt` builds the place-specific JSON prompt. Edit `prompt` in the next cell if you want a one-off experiment.

In [3]:
prompt = load_place_prompt(PLACE_ID)
print(prompt)

# CCTV Camera Image Analysis Prompt

## Role
You are an expert surveillance analyst tasked with analyzing CCTV camera footage from Hybernská street in Prague (urban street).

## Task
Analyze the provided CCTV camera image(s) from Hybernská street in Prague and extract traffic congestion, parked versus moving vehicles, construction, traffic-rule violations, weather, time of day, incidents, and other noteworthy observations about the scene. If multiple images are provided, it can be from more camera angles at the same time or a sequence of frames from one camera or a combination of both.

## Place-specific context
- High-angle municipal CCTV (camera 101048) looking along Hybernská toward the Powder Tower (Prašná brána) in the far distance; typical PRAHA / PRAGUE watermark in the corner
- A long urban street, not a signalized junction: judge flow on the remaining live carriageway, not on curb parking
- The right side of the view is often a construction enclosure (white metal fencing, red-

## Single image

Runs the first selected frame. Skip this cell if you only want a multi-image call.

In [4]:
single = analyze_camera_image(
    image_paths[0],
    prompt=prompt,
    parse_json=True,
    max_tokens=1500,
)
if "error" not in single:
    saved = save_experiment(single, PLACE_ID, kind="single")
    print(f"Saved {saved}")
single

Analyzing 1 image(s): camera_101048_20260907_143634_894.jpg


{'success': True,
 'analysis': {'congestion_level': 4,
  'traffic_flow': 'moderate',
  'vehicles': {'cars': 8,
   'vans': 0,
   'trucks': 1,
   'buses': 0,
   'trams': 0,
   'motorcycles': 0,
   'bicycles': 0},
  'parked_vehicles': 10,
  'pedestrians': 4,
  'construction': {'present': True,
   'occupies': 'both',
   'description': 'White metal fencing, red-and-white bollards, paving materials, and a site container occupy the right side of the roadway and sidewalk, leaving a narrowed contraflow passage.'},
  'violations': [],
  'weather': 'Dry pavement with partly cloudy daylight and good visibility',
  'time_of_day': 'afternoon',
  'incidents': 'none',
  'interesting': 'The works create a narrow two-way pinch with temporary yellow lane markings. Traffic is passing through, but the available carriageway is substantially narrowed. The Powder Tower is visible in the distance.',
  'scene_description': 'Urban street view toward the Powder Tower with moderate moving traffic, several curb-par

## Multiple images

Sends every selected frame in one request (several camera angles, or a short sequence).

In [5]:
batch = analyze_images(
    image_paths,
    prompt=prompt,
    parse_json=True,
    max_tokens=2000,
)
batch

Analyzing 5 image(s): camera_101048_20260907_143634_894.jpg, camera_101048_20260907_143637_019.jpg, camera_101048_20260907_143640_721.jpg, camera_101048_20260907_143642_802.jpg, camera_101048_20260907_143644_903.jpg


{'success': True,
 'analysis': {'congestion_level': 3,
  'traffic_flow': 'light',
  'vehicles': {'cars': 9,
   'vans': 2,
   'trucks': 1,
   'buses': 0,
   'trams': 0,
   'motorcycles': 0,
   'bicycles': 0},
  'parked_vehicles': 25,
  'pedestrians': 8,
  'construction': {'present': True,
   'occupies': 'both',
   'description': 'White metal fencing, red-and-white bollards, construction materials, paving debris, and a site container occupy the right side of the street, narrowing the carriageway and restricting the sidewalk.'},
  'violations': [],
  'weather': 'Overcast or partly cloudy, dry pavement, good visibility, no visible precipitation.',
  'time_of_day': 'afternoon',
  'incidents': 'none',
  'interesting': 'The road is narrowed by a construction pinch and temporary yellow lane markings, but vehicles continue through with visible gaps and no major queue. Numerous cars are parked along both curbs, especially beyond the works area. The Powder Tower is visible in the distance.',
  's

In [6]:
result = batch if "batch" in globals() else single

if result.get("error"):
    print(result["error"])
else:
    saved = save_experiment(result, PLACE_ID, kind="latest")
    print("saved:", saved)
    print("model:", result.get("model"))
    print("usage:", result.get("usage"))
    print()
    analysis = result.get("analysis")
    if isinstance(analysis, dict):
        for key, value in analysis.items():
            print(f"{key}: {value}")
    else:
        print(analysis)

model: gpt-5.6-luna
usage: {'completion_tokens': 667, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 344, 'rejected_prediction_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 10, 'engine_ttft_ms': 573, 'engine_ttlt_ms': 7450, 'pre_inference_ms': 163, 'service_tbt_ms': 10, 'service_ttft_ms': 1630, 'service_ttlt_ms': 8500, 'user_visible_ttft_ms': 1467}, 'prompt_tokens': 3827, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'total_tokens': 4494}

congestion_level: 3
traffic_flow: light
vehicles: {'cars': 9, 'vans': 2, 'trucks': 1, 'buses': 0, 'trams': 0, 'motorcycles': 0, 'bicycles': 0}
parked_vehicles: 25
pedestrians: 8
construction: {'present': True, 'occupies': 'both', 'description': 'White metal fencing, red-and-white bollards, construction materials, paving debris, and a site container occupy the right side of the street, narrowing the carriageway and restricting the sidewalk.'}
violations: []
weather: Over